# 2 · One question through the runtime: `rag_runtime.answer()`

This notebook follows **one question** through every step of `src/rag_runtime.py`, calling the real functions and
printing what each returns. The agent (notebook 3) calls exactly this pipeline through its tools.

```
question ─► validate (IDs, product, country) ─► is anything in effect? (code)
         ─► build_filters (SQL WHERE, by code) ─► embed_query (Vertex AI)
         ─► hybrid_search (BigQuery VECTOR_SEARCH, meaning + exact words, on filtered rows only)
         ─► collapse_same_hash ─► rerank (Vertex AI Ranking API) ─► authority_order (code)
         ─► add_neighbors ─► build_cards (numbered evidence, headers written by code)
         ─► sufficiency ─► generate (Gemini, JSON + claims) ─► checks: citations · numbers · claims · Check Grounding
            (any failure → ONE repair try with the checker's words, then everything is checked again)
         ─► the reply, assembled by code: answer + Sources + fixed notes ─► agent_log row
```

`LIVE = False` replays the responses recorded for the question below (tests/fixtures/runtime), so every step shows
real data at no cost. With `LIVE = True` you can change `QUESTION` and `COUNTRY` to anything.

In [1]:
LIVE = False   # False: recorded responses and local files, no cost. True: real Google Cloud calls (small cost).

from nb_setup import setup, short, table, module, functions
SUBMISSION = setup(live=LIVE)

QUESTION, COUNTRY, MODE = "Current AX-17 renal guidance in Germany?", "Germany", "current"   # a recorded question

submission : /Users/vikassingh/Downloads/GIT/GCP_FDE/Session-08-UseCases-Assignments/submission
mode       : OFFLINE: recorded responses, no cost
project    : tredence-fde-1 · answers: gemini-3.5-flash-lite · agent: gemini-3.1-flash-lite · embeddings: gemini-embedding-001


## What each service does, and the Databricks equivalent

| Step | On GCP (this project) | On Databricks |
|---|---|---|
| Hard filters + hybrid search | BigQuery SQL `WHERE` inside `VECTOR_SEARCH` (vector + lexical) | Mosaic AI Vector Search with `filters` and `query_type="HYBRID"` |
| Question embedding | Vertex AI `gemini-embedding-001`, task `RETRIEVAL_QUERY` | the same embedding endpoint used for the chunks |
| Reranking | Vertex AI Ranking API (`semantic-ranker-default`) | the Vector Search reranker, or a reranker model on Model Serving |
| Answer writer | Gemini (JSON output) | a Foundation Model API endpoint (or an external-model endpoint for Gemini) |
| Grounding check | Vertex AI Check Grounding | an LLM judge (MLflow / Agent Evaluation "groundedness"), or a custom check |
| Question log | BigQuery `agent_log` | a Delta table, or MLflow traces / inference tables |

Everything between the service calls (filters, authority order, evidence cards, the checks, the reply) is plain Python.

In [2]:
import rag_runtime as rt
functions(rt)

,line,function,what it does
0,79,_no_cloud_offline,Offline mode must never reach Google Cloud; fa...
1,85,_bq,
2,92,_genai,
3,105,with_retry,"Retry quota / transient errors (429, 503) with..."
4,126,doc_label,"'DE AX17 Current Label v4.2', not 'v4.2 v4.2'."
...,...,...,...
65,1303,cross_check,Data vs data: each registry rate must appear i...
66,1318,differences,"Differences and rankings, calculated by code s..."
67,1355,compare,Compare adverse-event evidence across studies ...
68,1449,format_trace,The trace the brief asks for: query → intent →...


## Step 0 · clean the inputs
`resolve_product` reads the product from the registry (only one exists, so `None` → AX-17); the country is normalised
to one spelling; the tool's **mode** fixes which statuses are allowed (the model never chooses a status).

In [3]:
from datetime import datetime, timezone
from policy import AUTHORITY_ORDER, GLOBAL_EVIDENCE_TYPES, MODES, normalize_country

product = rt.resolve_product(None)
country = normalize_country(COUNTRY)
now = datetime.now(timezone.utc)
print("product :", product, "(from the registry)")
print("country :", country)
print("mode    :", MODE, "→ statuses allowed:", MODES[MODE])

product : AX-17 (from the registry)
country : Germany
mode    : current → statuses allowed: ['APPROVED']


## Step 1 · validate against the registry
Nothing that does not exist is searched: an unknown document ID or an uncovered country stops here with a clear reply.

In [4]:
known_ids, unknown_ids = rt.ids_in_question(QUESTION)
print("IDs in the question → known:", known_ids, "· unknown:", unknown_ids)
print("covered countries:", rt.covered_countries())
table([{k: d[k] for k in ("document_id", "status", "countries", "effective_from", "valid_to", "superseded_by")}
       for d in rt.registry().values()])

IDs in the question → known: [] · unknown: []
covered countries: ['Germany', 'Singapore', 'UAE', 'United States']


,document_id,status,countries,effective_from,valid_to,superseded_by
0,DE-AX17-LBL-042,APPROVED,[Germany],2026-09-15 00:00:00+00:00,NaT,NaN
1,US-AX17-PI-050,APPROVED,[United States],2026-08-01 00:00:00+00:00,NaT,NaN
2,SB-AX17-2026-09,APPROVED,"[Germany, Singapore, UAE]",2026-10-07 09:00:00+00:00,NaT,NaN
3,HN-204,APPROVED,[GLOBAL],2025-06-12 00:00:00+00:00,NaT,NaN
4,HN-311,APPROVED,[GLOBAL],2026-01-18 00:00:00+00:00,NaT,NaN
5,HN-402,APPROVED,[GLOBAL],2026-05-22 00:00:00+00:00,NaT,NaN
6,DE-AX17-LBL-038,SUPERSEDED,[Germany],2025-02-03 00:00:00+00:00,2026-09-15 00:00:00+00:00,DE-AX17-LBL-042
7,DE-AX17-HSL-2019-2021,SUPERSEDED,[Germany],2019-01-01 00:00:00+00:00,2022-01-01 00:00:00+00:00,NaN


## Step 2 · is any local document in effect now? (D39, plain Python on the registry)

In [5]:
doc_ids = sorted(set(known_ids))
local = [d for d in rt.registry().values()
         if country in d["countries"] and d["product"] == product and d["status"] in MODES[MODE]
         and d["effective_from"] <= now and (d["valid_to"] is None or d["valid_to"] > now)]
for d in local:
    print(f"in effect: {d['document_id']:18} {d['status']:9} since {d['effective_from']:%Y-%m-%d}")

in effect: DE-AX17-LBL-042    APPROVED  since 2026-09-15
in effect: SB-AX17-2026-09    APPROVED  since 2026-10-07


## Step 3 · `build_filters`: the hard filters become a SQL `WHERE`, applied **before** the search
Status, country, product, effective dates, template exclusion and the embedding model are fixed by code, so the
superseded v3.8 and the US label **cannot be found** for this question.

In [6]:
where, params, readable = rt.build_filters(MODE, country, product, None, doc_ids, None)
print("WHERE", where.replace(" AND ", "\n  AND "))
print("\nparameters:", {p.name: (p.values if hasattr(p, "values") else p.value) for p in params})

WHERE product = @product
  AND status IN UNNEST(@statuses)
  AND boilerplate = FALSE
  AND embedding_model = @model
  AND effective_from <= @at
  AND (valid_to IS NULL OR valid_to > @at)
  AND (@country IN UNNEST(countries) OR (document_type IN UNNEST(@global_types)
  AND @global IN UNNEST(countries)))

parameters: {'product': 'AX-17', 'statuses': ['APPROVED'], 'model': 'gemini-embedding-001@768', 'at': datetime.datetime(2026, 10, 20, 14, 25, 2, 508308, tzinfo=datetime.timezone.utc), 'country': 'Germany', 'global_types': ['STUDY'], 'global': 'GLOBAL'}


## Step 4 · `embed_query`: the question becomes 768 numbers (Vertex AI)

In [7]:
vector = rt.embed_query(QUESTION)
print(len(vector), "numbers, first five:", [round(v, 4) for v in vector[:5]])

768 numbers, first five: [-0.0116, 0.007, 0.0074, -0.0654, 0.0008]


## Step 5 · `hybrid_search`: BigQuery `VECTOR_SEARCH`, meaning + exact words, on the filtered rows only
Two rankings of the allowed rows are merged: by **meaning** (the 768 numbers) and by **words** (BM25). In a hybrid
search the `distance` column is the **merged rank** (rank fusion: 1 − Σ 1/(60 + rank)), not a closeness: for these 18
rows it can only lie between 1 − 2/61 = 0.967 and 1 − 2/78 = 0.974. Checked against a meaning-only search, which
returns real cosine distances (0.23–0.42 for this question). Use it to order the candidates, never as a threshold.

In [8]:
key = {"question": QUESTION, "mode": MODE, "country": country, "as_of": None, "doc_ids": doc_ids,
       "document_type": None, "product": product, "model": rt.EMBEDDING_MODEL_TAG}   # the record/replay key
candidates, method = rt.hybrid_search(QUESTION, vector, where, params, key)
print("method:", method, "·", len(candidates), "candidates")
table([{"chunk_id": c["chunk_id"], "document": c["document_id"], "status": c["status"],
        "page": rt.page_label(c), "distance": round(c["distance"], 3)} for c in candidates])

method: hybrid (vector + lexical) · 18 candidates


,chunk_id,document,status,page,distance
0,DE-AX17-LBL-042-c0025,DE-AX17-LBL-042,APPROVED,p.11,0.968
1,DE-AX17-LBL-042-c0000,DE-AX17-LBL-042,APPROVED,p.1,0.968
2,SB-AX17-2026-09-c0008-t1,SB-AX17-2026-09,APPROVED,section: HelixNova Safety Bulletin SB-AX17-202...,0.968
3,SB-AX17-2026-09-c0000,SB-AX17-2026-09,APPROVED,section: HelixNova Safety Bulletin SB-AX17-202...,0.969
4,HN-204-c0000,HN-204,APPROVED,p.1–6,0.971
5,HN-311-c0020,HN-311,APPROVED,p.11,0.971
6,DE-AX17-LBL-042-c0012,DE-AX17-LBL-042,APPROVED,p.6,0.971
7,HN-204-c0025-t1,HN-204,APPROVED,p.13,0.971
8,SB-AX17-2026-09-c0036-f1,SB-AX17-2026-09,APPROVED,section: HelixNova Safety Bulletin SB-AX17-202...,0.972
9,HN-311-c0019-t1,HN-311,APPROVED,p.11,0.972


## Step 6–8 · collapse copies, rerank, then authority order
The Ranking API scores **relevance** only; it knows nothing about legal status. Code then orders by authority tier
(bulletin › label › study) and keeps the score inside each tier.

In [9]:
collapsed = rt.collapse_same_hash(candidates)
ranked = rt.rerank(QUESTION, collapsed, key)
ordered = rt.authority_order(ranked)
print(len(candidates), "→", len(collapsed), "after collapsing copies →", len(ranked), "kept by the Ranking API")
table([{"chunk_id": c["chunk_id"], "authority": c["authority"],
        "tier": AUTHORITY_ORDER.get(c["authority"], 9), "rank score": c["rank_score"]} for c in ordered])

18 → 18 after collapsing copies → 5 kept by the Ranking API


,chunk_id,authority,tier,rank score
0,SB-AX17-2026-09-c0000,SAFETY_BULLETIN,0,0.8810
1,DE-AX17-LBL-042-c0012,REGULATORY_LABEL,1,0.9506
2,DE-AX17-LBL-042-c0000,REGULATORY_LABEL,1,0.9430
3,DE-AX17-LBL-042-c0025,REGULATORY_LABEL,1,0.6917
4,HN-204-c0000,CLINICAL_STUDY,2,0.3535


## Step 9–10 · neighbours, then the evidence cards: **all that Gemini sees**
Each card has a short ID (`[E1]`), a header written by code (document, version, status, dates, page) and a ROLE
(LOCAL GUIDANCE, SUPPORTING EVIDENCE, HISTORICAL…). Gemini cites the IDs; code checks them.

In [10]:
evidence = rt.add_neighbors(ordered)
cards = rt.build_cards(evidence, now)
cards += [rt.registry_card(d, len(cards) + i) for i, d in enumerate(doc_ids, start=1)]
print(len(evidence), "evidence items ·", len(cards), "cards · roles:", [c["role"] for c in cards])
print("\n" + cards[0]["text"][:900])

5 evidence items · 5 cards · roles: ['LOCAL GUIDANCE', 'LOCAL GUIDANCE', 'LOCAL GUIDANCE', 'LOCAL GUIDANCE', 'SUPPORTING EVIDENCE (global study), NOT LOCAL GUIDANCE']

[E1] SB-AX17-2026-09 · SB AX17 2026 09 Safety Bulletin v1.0 · SAFETY_BULLETIN · Germany, Singapore, UAE · APPROVED · in effect 2026-10-07
     ROLE: LOCAL GUIDANCE · section: HelixNova Safety Bulletin SB-AX17-2026-09
# HelixNova Safety Bulletin SB-AX17-2026-09

Status: APPROVED - HIGH AUTHORITY Effective: 07-Oct-2026 10:00 CET Products: AX-17 Jurisdictions: Germany, Singapore, UAE Subject: Temporary hepatic monitoring enhancement Baseline hepatic laboratory values are required with repeat testing between Day 14 and Day 21. This bulletin does not change the approved German renal starting dose of 25 mg once daily for moderate renal impairment.


## Step 11 · `sufficiency`: the last gate before Gemini
A country question must have at least one LOCAL GUIDANCE card; otherwise the reply is "I don't have this", by code.

In [11]:
require_local = bool(country) and not doc_ids
ok, reasons = rt.sufficiency(QUESTION, cards, country, require_local)
print("enough evidence:", ok, "· reasons:", reasons)

enough evidence: True · reasons: []


## Step 12 · `generate`: Gemini writes JSON from the cards only
The prompt is the RULES (only these cards, cite each fact, today's date, never confirm a user's assumption) + the cards.
The JSON also lists its **claims**: one per number, with the card, and for a table the row and column it read.

In [12]:
import json

print(rt.RULES[:900], "…\n")
gen = rt.generate(QUESTION, MODE, country, cards, product)
print(json.dumps(gen, indent=1, ensure_ascii=False)[:1500])

You answer questions from {organisation} about {product} using ONLY the evidence cards below.

Rules:
- Every statement must come from the evidence. Put the card ID in square brackets after each claim, e.g. [E2].
- Copy numbers, doses, units and abbreviations exactly as written in the evidence. Never calculate or infer a value.
- ROLE decides how evidence may be used: LOCAL GUIDANCE is the answer for the asked country. SUPPORTING EVIDENCE
  (global study) may support but never replaces local guidance. HISTORICAL content must be described as superseded,
  with its dates, and never as current.
- REGISTRY FACTS cards hold the document register's facts (status, version, dates, which document superseded it);
  cite them for such facts.
- STRUCTURED DATA cards hold rows from the registry database (same values as the study reports); cite them
  like documents. COMPUTED BY CODE cards hold differ …

{
 "verdict": "answered",
 "answer": "For normal renal function (eGFR >=90), the starting dose i

## Step 13 · the checks, all by code except Check Grounding
1. every `[E#]` must be a card Gemini was given; 2. every number must appear in a card (a number the user assumed must
be rejected, not confirmed); 3. **the claim check**: every number is claimed, and every table value sits in the exact
cell its row and column name (`table_facts`); 4. Vertex AI Check Grounding scores how well each claim is supported
(bar 0.6). If any check fails, Gemini gets **one repair try** with the checker's words, then everything runs again;
a second failure is blocked.

In [13]:
passed, why = rt.check_citations(gen, cards)
cited = rt.cited_ids(gen)
numbers_ok, number_check = rt.check_numbers(gen["answer"], cards, cited, QUESTION)
claims_ok, claim_check = rt.check_claims(gen, cards, QUESTION)
g = rt.grounding(gen["answer"], cards)
print("citations :", passed, why, "· cited:", cited)
print("numbers   :", numbers_ok, number_check)
print("claims    :", claims_ok, claim_check)
print("grounding : support", g["score"], "(bar", rt.GROUNDING_THRESHOLD, ")")
for c in g["claims"]:
    print("   ", "✓" if c["supported"] else "✗", short(c["text"], 110))

citations : True ok · cited: ['E1', 'E2', 'E3']
numbers   : True pass
claims    : True pass (12 claims, 12 checked against their table cell)
grounding : support 0.982 (bar 0.6 )
    ✓ For normal renal function (eGFR >=90), the starting dose is 50 mg OD with a maximum of 100 mg/day and routine  …
    ✓ For mild renal impairment (eGFR 60-89), the starting dose is 50 mg OD with a maximum of 100 mg/day and routine …
    ✓ For moderate renal impairment (eGFR 30-59), the starting dose is 25 mg OD with a maximum of 50 mg/day, and mon …
    ✓ For severe renal impairment (eGFR 15-29), the starting dose is specialist review with a maximum of 25 mg/day a …


## Step 14 · the reply is assembled by **code** around Gemini's text
Fixed lines first (a premise the user assumed), then the answer, the Sources block (built from the cited cards, never
by Gemini), and fixed notes (an earlier version exists; or, for historical answers, the current version).

In [14]:
print("premise lines :", rt.premise_notes(QUESTION, cards, gen["answer"]))
print("partial note  :", rt.partial_note(gen))
print("earlier version (current answers):", rt.earlier_version_notes(cited, cards))
print("\n" + rt.sources_block(cited, cards))

premise lines : []
partial note  : []
earlier version (current answers): ['Earlier version: DE-AX17-LBL-038 (DE AX17 Superseded Label v3.8), superseded on 2026-09-15. Ask for the historical guidance to see it.']

Sources
[E1] SB-AX17-2026-09 · SB AX17 2026 09 Safety Bulletin v1.0 · APPROVED · effective 2026-10-07 · section: HelixNova Safety Bulletin SB-AX17-2026-09
     https://storage.cloud.google.com/tredence-fde-1-medevidence-vks/raw/SB_AX17_2026_09_Safety_Bulletin.docx
[E2] DE-AX17-LBL-042 · DE AX17 Current Label v4.2 · APPROVED · effective 2026-09-15 · p.6, Table 3
     https://storage.cloud.google.com/tredence-fde-1-medevidence-vks/raw/DE_AX17_Current_Label_v4.2.pdf#page=6
[E3] DE-AX17-LBL-042 · DE AX17 Current Label v4.2 · APPROVED · effective 2026-09-15 · p.1
     https://storage.cloud.google.com/tredence-fde-1-medevidence-vks/raw/DE_AX17_Current_Label_v4.2.pdf#page=1


## Step 15 · the whole thing in one call: `rt.answer()`
The same steps, plus the log row (offline: one line in `output/agent_log_offline.jsonl`; live: a BigQuery `agent_log` row).

In [15]:
r = rt.answer(QUESTION, mode=MODE, country=country, caller="notebook")
for k in ("outcome", "search_method", "citation_check", "number_check", "claim_check", "grounding_check", "repair", "latency_ms"):
    print(f"{k:16} {r.get(k)}")
if not LIVE:
    print("(offline: every service call is replayed from a file, so the stage times are ~0 ms)")
print("\nWHAT THE USER SEES:\n")
print("\n\n".join(r.get("lead_notes", []) + [r["answer"]] + ([r["sources"]] if r.get("sources") else []) + r.get("notes", [])))

outcome          answered
search_method    hybrid (vector + lexical)
citation_check   pass
number_check     pass
claim_check      pass (12 claims, 12 checked against their table cell)
grounding_check  pass: support 0.98
repair           None
latency_ms       {'embed': 1, 'search': 0, 'rerank': 0, 'generate': 0, 'grounding': 0, 'total': 4}
(offline: every service call is replayed from a file, so the stage times are ~0 ms)

WHAT THE USER SEES:

For normal renal function (eGFR >=90), the starting dose is 50 mg OD with a maximum of 100 mg/day and routine monitoring [E2]. For mild renal impairment (eGFR 60-89), the starting dose is 50 mg OD with a maximum of 100 mg/day and routine monitoring [E2]. For moderate renal impairment (eGFR 30-59), the starting dose is 25 mg OD with a maximum of 50 mg/day, and monitoring requires Recheck 2-4 weeks [E1, E2, E3]. For severe renal impairment (eGFR 15-29), the starting dose is specialist review with a maximum of 25 mg/day and enhanced monitoring [E2].


## The other entry points (all recorded, so they run offline)

| Function | Used by the tool | What is different |
|---|---|---|
| `answer(mode="historical", as_of=…)` | `search_historical` | statuses APPROVED + SUPERSEDED, in effect on the date; adds "Current version: …" (D71) |
| `lookup(document_id, table=…)` | `lookup_document` | the exact ID restricts the search to one document; a named table is shown word for word |
| `compare(study_ids, …)` | `compare_studies` | verified SQL + each study's own table; differences computed by code |
| `text2sql.ask(question)` | (the `sql_analyst` agent's tools) | SQL on two curated views, checked and dry-run before it runs |

In [16]:
h = rt.answer("What was the renal starting dose for moderate impairment in Germany?", mode="historical",
             country="Germany", as_of="2025-06-30", caller="notebook")
print(h["outcome"], "·", h["answer"][:300], "\n")
print("notes:", h.get("notes"))

answered · The historical starting dose for moderate renal impairment in Germany was 50 mg OD, effective from 03-Feb-2025 to 15-Sep-2026 [E1, E2]. 

notes: ['Current version: DE-AX17-LBL-042 (DE AX17 Current Label v4.2), in effect since 2026-09-15. Ask for the current guidance to see it.']


In [17]:
lk = rt.lookup("HN-204", table="Table 14", caller="notebook")
print(lk["outcome"], "·", lk["citation_check"], "\n")
print(lk["answer"][:900])

answered · verbatim table (no generation) 

Table 14. Treatment-emergent adverse events · HN-204 · HN-204 Clinical Study v1.0 · APPROVED · p.11

| Adverse event | AX-17 % | Control % | Serious? |
|-|-|-|-|
| Nausea | 12.7 | 6.0 | No |
| Headache | 14.0 | 6.6 | No |
| Fatigue | 15.3 | 7.2 | No |
| Dizziness | 16.6 | 7.8 | No |
| Rash | 17.8 | 8.4 | No |
| Transaminase elevation | 19.1 | 9.0 | Yes |

Source: HN-204 · HN-204 Clinical Study v1.0 · APPROVED · effective 2025-06-12 · p.11, Table 14
     https://storage.cloud.google.com/tredence-fde-1-medevidence-vks/raw/HN-204_Clinical_Study.pdf#page=11


In [18]:
cp = rt.compare(["HN-204", "HN-311"], event="", question="Compare AE evidence in HN-204 and HN-311.", caller="notebook")
print(cp["outcome"], "\n")
print(cp["answer"][:1200])

answered 

In HN-204, AX-17 treatment percentages for adverse events are 12.7% for nausea [E1, E3], 14.0% for headache [E1, E3], 15.3% for fatigue [E1, E3], 16.6% for dizziness [E1, E3], 17.8% for rash [E1, E3], and 19.1% for transaminase elevation [E1, E3]. Transaminase elevation is serious [E1]. In HN-311, AX-17 treatment percentages are 11.8% for nausea [E2, E3], 13.0% for headache [E2, E3], 14.2% for fatigue [E2, E3], 15.4% for dizziness [E2, E3], 16.6% for rash [E2, E3], and 17.7% for transaminase elevation [E2, E3]. Transaminase elevation is serious [E2]. Control percentages for all adverse events are identical in both studies at 6.0% for nausea [E1, E2, E3], 6.6% for headache [E1, E2, E3], 7.2% for fatigue [E1, E2, E3], 7.8% for dizziness [E1, E2, E3], 8.4% for rash [E1, E2, E3], and 9.0% for transaminase elevation [E1, E2, E3]. Comparing treatment percentages (HN-311 minus HN-204), differences are -0.9 points for nausea [E4], -1.0 points for headache [E4], -1.1 points for fatig

In [19]:
import text2sql
sq = text2sql.ask("Which study had the highest rash rate?", caller="notebook")
print(sq["outcome"], "\n")
print(sq["answer"])

answered 

Result of a read-only query on v_adverse_events (1 row):

| study_id | rash_rate |
|-|-|
| HN-402 | 19.7 |

SQL:
SELECT study_id, treatment_rate_pct AS rash_rate FROM `tredence-fde-1.medevidence_v1_vks.v_adverse_events` WHERE LOWER(event) = 'rash' ORDER BY treatment_rate_pct DESC LIMIT 1


## Try your own question (LIVE only)
Set `LIVE = True` in the first cell, re-run it, then change the question here. Recorded responses exist only for the
questions above, so offline this cell only explains itself.

In [20]:
MY_QUESTION, MY_COUNTRY, MY_MODE = "What hepatic monitoring does the safety bulletin require?", "Germany", "current"
if LIVE:
    mine = rt.answer(MY_QUESTION, mode=MY_MODE, country=MY_COUNTRY, caller="notebook")
    print(mine["outcome"], "·", mine["grounding_check"], "\n")
    print("\n\n".join([mine["answer"]] + ([mine["sources"]] if mine.get("sources") else []) + mine.get("notes", [])))
else:
    print("Offline: set LIVE = True in the first cell to ask your own question (about one cent per question).")

Offline: set LIVE = True in the first cell to ask your own question (about one cent per question).
